# 00 Foundations — Calculus Refresher

**Status:** Complete

**Learning outcome:** Implement forward and backward pass of a scalar function manually. Verify against sympy symbolic derivative.

## Why This Matters

Every neural network is trained by computing derivatives. Backpropagation is just the chain rule applied recursively through a computation graph. If you can't differentiate a scalar function by hand and verify it numerically, the rest of this repository will be opaque.

This notebook builds the core skill: compute a derivative analytically, verify it with finite differences, and extend to multivariate functions (gradients). This is the foundation that autograd (Section 01) automates.

## Theory Sketch

The derivative of $f$ at $x$ is defined as:

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

In practice we approximate with a small $h$ (finite difference). The **central difference** is more accurate:

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

The **chain rule** for composed functions $f(g(x))$:

$$\frac{d}{dx}f(g(x)) = f'(g(x)) \cdot g'(x)$$

For multivariate functions $f(x_1, x_2, \ldots)$, the **gradient** is:

$$\nabla f = \left[\frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}, \ldots\right]$$

## From-Scratch Implementation (NumPy)

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sympy as sp

# --- 1. Numerical gradient via central difference ---
def numerical_gradient(f, x, h=1e-5):
    """Compute derivative of f at x using central difference."""
    return (f(x + h) - f(x - h)) / (2 * h)

# --- 2. Analytic gradient for f(x) = x^3 + 2x^2 ---
def f(x):
    return x**3 + 2*x**2

def analytic_gradient(x):
    """f'(x) = 3x^2 + 4x (derived by hand)."""
    return 3*x**2 + 4*x

# --- 3. Compare at several points ---
test_points = [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
print(f"{'x':>6} {'analytic':>12} {'numerical':>12} {'diff':>12}")
print("-" * 46)
for x in test_points:
    a = analytic_gradient(x)
    n = numerical_gradient(f, x)
    print(f"{x:6.1f} {a:12.6f} {n:12.6f} {abs(a-n):12.2e}")
    assert abs(a - n) < 1e-4, f"Gradient mismatch at x={x}"

print("\nAll gradients match to 4 decimal places.")

     x     analytic    numerical         diff
----------------------------------------------
  -2.0     4.000000     4.000000     1.59e-10
  -1.0    -1.000000    -1.000000     9.34e-11
   0.0     0.000000     0.000000     1.00e-10
   1.0     7.000000     7.000000     1.12e-10
   2.0    20.000000    20.000000     3.09e-10
   3.0    39.000000    39.000000     3.00e-10

All gradients match to 4 decimal places.


---
**Understanding Derivatives and Finite Differences**

**Plain language:** A derivative tells you how fast something is changing at a specific moment — like the speedometer in your car tells you your speed right now, not your average speed over the trip. We can approximate it by checking two points very close together: how much did the output change for a tiny nudge in the input? That ratio is the derivative.

**Building intuition:** The central difference formula $\frac{f(x+h) - f(x-h)}{2h}$ measures the slope of a line through two points straddling $x$. As $h$ shrinks, this slope approaches the true derivative. The "central" version (symmetric around $x$) cancels out the leading error term, making it $O(h^2)$ accurate instead of $O(h)$. In neural networks, numerical gradients serve as a debugging tool: if your analytic gradient disagrees with the numerical one, your math (or code) has a bug.

**Formal statement:** The derivative $f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$ is approximated by the central difference $f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} + O(h^2)$. For $f(x) = x^3 + 2x^2$, by the power rule: $f'(x) = 3x^2 + 4x$. The approximation error is bounded by $\frac{h^2}{6} \max|f'''|$ on the interval.

---

In [2]:
# --- 4. Verify against sympy symbolic derivative ---
x_sym = sp.Symbol('x')
f_sym = x_sym**3 + 2*x_sym**2
f_prime_sym = sp.diff(f_sym, x_sym)
print(f"f(x)  = {f_sym}")
print(f"f'(x) = {f_prime_sym}")

# Verify sympy matches our analytic derivative
for x_val in test_points:
    sympy_val = float(f_prime_sym.subs(x_sym, x_val))
    hand_val = analytic_gradient(x_val)
    assert abs(sympy_val - hand_val) < 1e-10, f"Sympy mismatch at x={x_val}"

print("Sympy symbolic derivative matches hand-derived formula exactly.")

f(x)  = x**3 + 2*x**2
f'(x) = 3*x**2 + 4*x
Sympy symbolic derivative matches hand-derived formula exactly.


In [3]:
# --- 5. Chain rule example: d/dx[sin(x^2)] ---
def g(x):
    return np.sin(x**2)

def g_analytic_grad(x):
    """d/dx[sin(x^2)] = cos(x^2) * 2x  (chain rule)."""
    return np.cos(x**2) * 2*x

print("Chain rule: d/dx[sin(x^2)] = cos(x^2) * 2x")
print(f"{'x':>6} {'analytic':>12} {'numerical':>12} {'diff':>12}")
print("-" * 46)
for x_val in [0.5, 1.0, 1.5, 2.0]:
    a = g_analytic_grad(x_val)
    n = numerical_gradient(g, x_val)
    print(f"{x_val:6.1f} {a:12.6f} {n:12.6f} {abs(a-n):12.2e}")
    assert abs(a - n) < 1e-4

Chain rule: d/dx[sin(x^2)] = cos(x^2) * 2x
     x     analytic    numerical         diff
----------------------------------------------
   0.5     0.968912     0.968912     4.21e-11
   1.0     1.080605     1.080605     2.39e-10
   1.5    -1.884521    -1.884521     3.98e-11
   2.0    -2.614574    -2.614574     9.81e-10


---
**Understanding the Chain Rule**

**Plain language:** Imagine a set of gears connected in a chain. If the first gear turns twice as fast as the handle, and the second gear turns three times as fast as the first, then the second gear turns six times as fast as the handle. The chain rule says: to find the total rate of change through a sequence of steps, multiply the individual rates together. This is exactly how backpropagation works — it multiplies local rates layer by layer.

**Building intuition:** For a composite function $f(g(x))$, the chain rule says "differentiate the outer function, then multiply by the derivative of the inner." Written as $\frac{df}{dx} = \frac{df}{dg} \cdot \frac{dg}{dx}$, it looks like fraction cancellation (though it's not — it's a theorem). In a neural network with 10 layers, backprop applies the chain rule 10 times, multiplying Jacobians at each step. If any of these factors is very small (<< 1), gradients vanish; if very large (>> 1), they explode. This is the core tension of deep learning.

**Formal statement:** If $y = f(g(x))$ where $f$ and $g$ are differentiable, then $\frac{dy}{dx} = f'(g(x)) \cdot g'(x)$. For multiple compositions: $\frac{d}{dx}f_n(f_{n-1}(\cdots f_1(x)\cdots)) = \prod_{i=1}^n f_i'(z_i)$ where $z_i$ is the input to $f_i$. This generalises to vector-valued functions via the Jacobian product $\frac{\partial \mathbf{y}}{\partial \mathbf{x}} = \prod_{i=1}^n J_i$.

---

In [4]:
# --- 6. Multivariate: gradient of f(x,y) = x^2*y + y^3 ---
def f_multi(xy):
    x, y = xy
    return x**2 * y + y**3

def numerical_gradient_multi(f, xy, h=1e-5):
    """Compute gradient of f at point xy using central difference."""
    grad = np.zeros_like(xy)
    for i in range(len(xy)):
        xy_plus = xy.copy(); xy_plus[i] += h
        xy_minus = xy.copy(); xy_minus[i] -= h
        grad[i] = (f(xy_plus) - f(xy_minus)) / (2 * h)
    return grad

# Analytic: df/dx = 2xy, df/dy = x^2 + 3y^2
point = np.array([2.0, 3.0])
analytic_grad = np.array([2*point[0]*point[1], point[0]**2 + 3*point[1]**2])
numerical_grad = numerical_gradient_multi(f_multi, point)

print(f"f(x,y) = x^2*y + y^3")
print(f"Point: ({point[0]}, {point[1]})")
print(f"Analytic  gradient: {analytic_grad}")
print(f"Numerical gradient: {numerical_grad}")
print(f"Max difference: {np.max(np.abs(analytic_grad - numerical_grad)):.2e}")
assert np.allclose(analytic_grad, numerical_grad, atol=1e-4)

f(x,y) = x^2*y + y^3
Point: (2.0, 3.0)
Analytic  gradient: [12. 31.]
Numerical gradient: [12. 31.]
Max difference: 2.56e-10


---
**Understanding Multivariate Gradients**

**Plain language:** When a function depends on multiple inputs (like a recipe depending on both temperature and cooking time), the gradient tells you how sensitive the result is to each input separately. It's a list of numbers — one per input — each saying "if you nudge this input a little, here's how much the output changes." In neural networks, each weight is an input, and the gradient tells the optimizer how to adjust every weight simultaneously.

**Building intuition:** The gradient $\nabla f = [\frac{\partial f}{\partial x_1}, \ldots, \frac{\partial f}{\partial x_n}]$ is a vector that points in the direction of steepest increase. Its magnitude tells you how steep the climb is. Gradient descent moves in the *opposite* direction: $\theta_{t+1} = \theta_t - \eta \nabla f(\theta_t)$. Each partial derivative $\frac{\partial f}{\partial x_i}$ treats all other variables as constants — this is why you can compute gradients for millions of weights independently. In a neural network with $N$ parameters, the gradient is an $N$-dimensional vector computed in one backward pass.

**Formal statement:** For $f: \mathbb{R}^n \to \mathbb{R}$, the gradient $\nabla f(x) \in \mathbb{R}^n$ has components $(\nabla f)_i = \frac{\partial f}{\partial x_i}$. It is the unique vector satisfying $f(x + \epsilon v) \approx f(x) + \epsilon \langle \nabla f(x), v \rangle$ for all directions $v$ and small $\epsilon$. The directional derivative in direction $v$ is $D_v f = \nabla f \cdot v$, maximised when $v = \nabla f / \|\nabla f\|$.

---

## PyTorch Rewrite

In [5]:
import torch

# Verify our analytic gradient using PyTorch autograd
x_t = torch.tensor(2.0, requires_grad=True)
y_t = x_t**3 + 2*x_t**2
y_t.backward()

pytorch_grad = x_t.grad.item()
hand_grad = analytic_gradient(2.0)
print(f"PyTorch autograd at x=2: {pytorch_grad}")
print(f"Hand-derived at x=2:     {hand_grad}")
assert abs(pytorch_grad - hand_grad) < 1e-4
print("PyTorch autograd matches hand-derived gradient.")

PyTorch autograd at x=2: 20.0
Hand-derived at x=2:     20.0
PyTorch autograd matches hand-derived gradient.


## Training Run

Not applicable for this foundations notebook — no model to train.

## Internal Probing

Visualise f(x) with tangent lines at three points to build geometric intuition for derivatives.

In [6]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot f(x)
x_plot = np.linspace(-3, 3, 200)
y_plot = x_plot**3 + 2*x_plot**2
ax.plot(x_plot, y_plot, 'b-', linewidth=2, label=r'$f(x) = x^3 + 2x^2$')

# Tangent lines at three points
tangent_points = [-2.0, 0.0, 1.5]
colors = ['red', 'green', 'orange']
for x0, color in zip(tangent_points, colors):
    y0 = f(x0)
    slope = analytic_gradient(x0)
    x_tang = np.linspace(x0 - 1.0, x0 + 1.0, 50)
    y_tang = y0 + slope * (x_tang - x0)
    ax.plot(x_tang, y_tang, '--', color=color, linewidth=1.5,
            label=f"tangent at x={x0} (slope={slope:.1f})")
    ax.plot(x0, y0, 'o', color=color, markersize=8)

ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Function with tangent lines — derivative as slope')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(-5, 30)
plt.tight_layout()
plt.show()
print("Tangent line visualisation complete.")

Tangent line visualisation complete.


/var/folders/sy/gz5gl6d91cbfwd2z85r62rbc0000gn/T/ipykernel_27313/2566218431.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Structured Interpretation

### Results
- Numerical (central difference) and analytic derivatives agree to <1e-9 across all test points
- Chain rule application verified: d/dx[sin(x²)] = cos(x²)·2x
- Multivariate gradient verified for f(x,y) = x²y + y³
- PyTorch autograd produces identical results to hand derivation

### Findings
- Central difference (h=1e-5) is accurate to ~1e-10, far exceeding our 1e-4 threshold
- The derivative is the slope of the tangent line — geometrically visible in the plot
- Numerical gradients are a reliable verification tool for any analytic derivative

### Implications
- `numerical_gradient` can be reused in later notebooks to verify backprop implementations
- The chain rule pattern (multiply local derivatives) is exactly what autograd automates (Section 01)
- Multivariate gradients are the foundation of weight updates in neural networks

### Considerations
- Finite difference breaks down for very small h (floating point cancellation) or very large h (poor approximation)
- h=1e-5 is a practical sweet spot for float64
- Central difference is O(h²) accurate vs forward difference O(h)